In [ ]:
#!/usr/bin/env python3
"""
NB12: Real Network Walking Distances
Computes true routing distances using OpenStreetMap (OSMNX) instead of Euclidean distances.
Demonstrates systematic overestimation of Euclidean coverage.
"""



# NB 12 — Real Walking Distances (OSM Network Analysis)

Baseline coverage calculations use Euclidean (Haversine) distance. However, actual
walking distances in urban environments are bounded by the road network.
This notebook downloads the OSM walking network for a sample region (e.g., Brussels)
and calculates network-based coverage vs Euclidean coverage.

**Expected runtime: ~2-5 minutes**


In [ ]:
Setup
import sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import osmnx as ox
import networkx as nx
from scipy.spatial.distance import cdist

PROJECT_ROOT = Path('.')
V3_DIR   = PROJECT_ROOT / 'mda_project' / 'data' / 'processed_v3'
OUT_DIR  = PROJECT_ROOT / 'mda_project' / 'data' / 'output'
FIG_DIR  = OUT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

EARTH_R = 6371.0088

def haversine_matrix(pts1, pts2):
    lat1, lon1 = np.radians(pts1[:, 0:1]), np.radians(pts1[:, 1:2])
    lat2, lon2 = np.radians(pts2[:, 0:1]).T, np.radians(pts2[:, 1:2]).T
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * EARTH_R * np.arctan2(np.sqrt(np.clip(a, 0, 1)), np.sqrt(1 - np.clip(a, 0, 1)))




In [ ]:
Load Data
mission = pd.read_parquet(V3_DIR / 'mission_records_v3.parquet')
aed = pd.read_parquet(V3_DIR / 'aed_records_v3.parquet')

# Filter a test region: Central Antwerp polygon to make OSM query fast
# Bounding box for Antwerp: [51.15, 51.25] Lat, [4.35, 4.45] Lon
lat_min, lat_max = 51.15, 51.25
lon_min, lon_max = 4.35, 4.45

m_reg = mission[(mission['latitude'] >= lat_min) & (mission['latitude'] <= lat_max) & 
                (mission['longitude'] >= lon_min) & (mission['longitude'] <= lon_max)].copy()
aed_reg = aed[(aed['latitude'] >= lat_min) & (aed['latitude'] <= lat_max) & 
              (aed['longitude'] >= lon_min) & (aed['longitude'] <= lon_max)].copy()

# Sample 500 missions for rendering speed
if len(m_reg) > 500:
    m_sample = m_reg.sample(500, random_state=42).reset_index(drop=True)
else:
    m_sample = m_reg.reset_index(drop=True)

print(f"Sampled {len(m_sample)} missions and {len(aed_reg)} AEDs in Antwerp box.")




In [ ]:
Download network
print("Downloading OSM walking network for Brussels bbox...")
G = ox.graph_from_bbox(bbox=(lat_max, lat_min, lon_max, lon_min), network_type='walk')
print(f"Graph loaded: {len(G.nodes)} nodes, {len(G.edges)} edges")

# Vectorized nearest node finding
# OSMnx node IDs
nodes_df = ox.graph_to_gdfs(G, edges=False)

print("Finding nearest network nodes...")
# Add nearest nodes to dataframe
m_nodes = ox.distance.nearest_nodes(G, m_sample['longitude'].values, m_sample['latitude'].values)
aed_nodes = ox.distance.nearest_nodes(G, aed_reg['longitude'].values, aed_reg['latitude'].values)

m_sample['node'] = m_nodes

# Map AED nodes to a set for fast checking
# Since we just want distance to nearest AED, we can use multi-source Dijkstra
aed_nodes_set = set(aed_nodes)

# Add dummy node connected to all AED nodes with 0 weight
G_mod = G.copy()
G_mod.add_node('SUPER_AED')
for an in aed_nodes_set:
    G_mod.add_edge('SUPER_AED', an, weight=0, length=0)
    # Undirected fallback
    G_mod.add_edge(an, 'SUPER_AED', weight=0, length=0)

print("Computing network distances...")
# Running single source Dijkstra from the SUPER_AED node gives the shortest path
# to ALL other nodes in the network. Super fast!
network_dists_dict = nx.single_source_dijkstra_path_length(G_mod, 'SUPER_AED', weight='length')

# Map back to missions
network_dists_m = [network_dists_dict.get(node, float('inf')) / 1000.0 for node in m_sample['node']]
m_sample['net_dist_km'] = network_dists_m

# Also calculate Haversine distance
aed_pts = aed_reg[['latitude', 'longitude']].values
m_pts = m_sample[['latitude', 'longitude']].values
h_dists = np.min(haversine_matrix(m_pts, aed_pts), axis=1)
m_sample['euc_dist_km'] = h_dists

# Ratio net/euc
m_sample['dist_ratio'] = m_sample['net_dist_km'] / m_sample['euc_dist_km']
m_sample.replace([np.inf, -np.inf], np.nan, inplace=True)
median_ratio = m_sample['dist_ratio'].dropna().median()

print(f"\nResults for N={len(m_sample)} sample in Antwerp:")
print(f"Median Euclidean Dist: {m_sample['euc_dist_km'].median():.3f} km")
print(f"Median Network Dist:   {m_sample['net_dist_km'].median():.3f} km")
print(f"Median Ratio (Net/Euc): {median_ratio:.2f}x")

eu_cov = (m_sample['euc_dist_km'] <= 0.5).mean() * 100
nw_cov = (m_sample['net_dist_km'] <= 0.5).mean() * 100
print(f"True 500m Coverage drops from {eu_cov:.1f}% (Euclidean) to {nw_cov:.1f}% (Network)")

m_sample.to_csv(FIG_DIR / 'table_12_network_distances.csv', index=False)




In [ ]:
Figure: Distance Bias Evaluation
print("Rendering Fig 9 (Network vs Euclidean Distance)...")
fig9, ax = plt.subplots(figsize=(8, 6), dpi=200)

ax.scatter(m_sample['euc_dist_km'], m_sample['net_dist_km'], alpha=0.5, color='#2980b9', edgecolors='white')
max_val = max(m_sample['euc_dist_km'].max(), m_sample['net_dist_km'].max()) * 1.05
ax.plot([0, max_val], [0, max_val], 'r--', lw=2, label='1:1 Line (Perfect Match)')
ax.plot([0, max_val], [0, max_val * median_ratio], 'k-', lw=1.5, label=f'Median Ratio ({median_ratio:.2f}x)')

# Coverage box
ax.axvspan(0, 0.5, color='gray', alpha=0.1, zorder=0)
ax.axhspan(0, 0.5, color='gray', alpha=0.1, zorder=0)

ax.set_title(f'(i) True Walking Distance vs Euclidean Distance\nSample: N={len(m_sample)} (Brussels). 500m Coverage: {eu_cov:.1f}% (Euc) → {nw_cov:.1f}% (Net)', fontweight='bold')
ax.set_xlabel('Euclidean (Haversine) Distance [km]')
ax.set_ylabel('OSM Network Walking Distance [km]')
ax.set_xlim(0, max_val)
ax.set_ylim(0, max_val)
ax.grid(ls=':', alpha=0.6)
ax.legend()

# Add text for the "Overestimation Zone"
ax.text(0.4, 0.8, "Overestimation\nZone\n(Euc < 500m, Net > 500m)", ha='center', va='center', rotation=90,
        color='#c0392b', fontsize=10, bbox=dict(facecolor='white', edgecolor='#c0392b', alpha=0.8))
# Box coordinates for the overestimation zone: euc in [0, 0.5], net in [0.5, max_val]
import matplotlib.patches as patches
rect = patches.Rectangle((0, 0.5), 0.5, max_val - 0.5, linewidth=1, edgecolor='#c0392b', facecolor='#c0392b', alpha=0.2)
ax.add_patch(rect)

fig9.tight_layout()
fig9.savefig(FIG_DIR / 'fig9_network_distance.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.close(fig9)
print("  -> saved fig9_network_distance.png")
print("\n=== NB12 Complete ===")

